In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Veri yükleme ve birleştirme
def load_and_combine_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Veri setlerini birleştirerek daha fazla veri ile çalışma
    combined_df = pd.concat([train_df, test_df], ignore_index=True)

    print(f"Toplam veri sayısı: {len(combined_df)}")
    print("\nVeri seti özeti:")
    print(combined_df.info())

    return combined_df

# Veri temizleme ve ön işleme
def clean_and_preprocess(df):
    # Eksik değer analizi
    print("\nEksik değerler:")
    print(df.isnull().sum())

    # Kategorik değişkenleri dönüştürme
    label_encoder = LabelEncoder()
    df['fuel_type'] = df['fuel_type'].fillna('Unknown')
    df['fuel_type'] = label_encoder.fit_transform(df['fuel_type'])

    # Kaza geçmişi - daha detaylı işleme
    df['accident'] = df['accident'].apply(
        lambda x: 1 if isinstance(x, str) and 'accident' in x.lower() else 0)

    # Temiz başlık - daha güvenli dönüşüm
    df['clean_title'] = df['clean_title'].map(
        lambda x: 1 if str(x).lower() == 'yes' else 0 if pd.notna(x) else 0)

    # Sayısal değişkenlerde aykırı değer temizleme
    numerical_cols = ['milage', 'price', 'model_year']
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Aykırı değerleri baskılama
        df[col] = np.where(df[col] < lower_bound, lower_bound,
                          np.where(df[col] > upper_bound, upper_bound, df[col]))

    # Motor bilgilerini çıkarma (daha gelişmiş)
    df['engine_power'] = df['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)
    df['engine_volume'] = df['engine'].str.extract(r'(\d+\.?\d*)L').astype(float)
    df['cylinders'] = df['engine'].str.extract(r'(\d+)\s*Cylinder').astype(float)

    # Eksik motor değerlerini ortalama ile doldurma
    df['engine_power'].fillna(df['engine_power'].median(), inplace=True)
    df['engine_volume'].fillna(df['engine_volume'].median(), inplace=True)
    df['cylinders'].fillna(df['cylinders'].median(), inplace=True)

    # Araç yaşını hesaplama
    df['age'] = 2023 - df['model_year']

    # Gereksiz sütunları çıkarma
    df.drop(['engine', 'transmission', 'ext_col', 'int_col'], axis=1, inplace=True)

    return df

# Özellik mühendisliği
def feature_engineering(df):
    # Marka-model etkileşimi
    df['brand_model'] = df['brand'] + '_' + df['model']

    # Kilometre başına fiyat (target encoding benzeri)
    brand_model_avg = df.groupby('brand_model')['price'].mean().to_dict()
    df['brand_model_avg_price'] = df['brand_model'].map(brand_model_avg)

    # Yakıt tipine göre ortalama fiyat
    fuel_avg = df.groupby('fuel_type')['price'].mean().to_dict()
    df['fuel_type_avg_price'] = df['fuel_type'].map(fuel_avg)

    # Kategorik değişkenleri one-hot encoding
    df = pd.get_dummies(df, columns=['brand', 'model'], drop_first=True)

    return df

# Model eğitimi ve değerlendirme
def train_and_evaluate(df):
    # Bağımlı ve bağımsız değişkenler
    X = df.drop(['price', 'id', 'brand_model'], axis=1)
    y = df['price']

    # Kategorik ve sayısal sütunları ayırma
    categorical_cols = [c for c in X.columns if X[c].dtype == 'object']
    numerical_cols = [c for c in X.columns if X[c].dtype in ['int64', 'float64']]

    # Ön işleme pipeline'ı
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols)])

    # Modeller
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(),
        'Lasso': Lasso(),
        'Random Forest': RandomForestRegressor(random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42)
    }

    # Model değerlendirme
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    results = []
    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        results.append({
            'Model': name,
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse,
            'R2': r2
        })

        # En iyi modeli kaydet
        if name == 'Gradient Boosting':
            joblib.dump(pipeline, 'best_model.pkl')

    # Sonuçları DataFrame'e dönüştür
    results_df = pd.DataFrame(results)
    print("\nModel Performansları:")
    print(results_df.sort_values(by='R2', ascending=False))

    # Performans görselleştirme
    plt.figure(figsize=(12, 6))
    sns.barplot(x='Model', y='R2', data=results_df)
    plt.title('Model Performans Karşılaştırması (R2 Skoru)')
    plt.xticks(rotation=45)
    plt.show()

    return results_df

# Tahmin fonksiyonu
def predict_price(input_data, model_path='best_model.pkl'):
    try:
        model = joblib.load(model_path)
        prediction = model.predict(input_data)
        return prediction[0]
    except Exception as e:
        print(f"Tahmin hatası: {str(e)}")
        return None

# Ana işlem akışı
def main():
    # Veri yükle ve işle
    df = load_and_combine_data('dataset/train.csv', 'dataset/test.csv')
    df = clean_and_preprocess(df)
    df = feature_engineering(df)

    # Model eğit ve değerlendir
    results = train_and_evaluate(df)

    # Kullanıcı etkileşimi
    print("\nAraç Fiyat Tahmini Uygulaması")
    print("-----------------------------")

    while True:
        try:
            print("\nAraç özelliklerini girin:")
            brand = input("Marka: ")
            model_name = input("Model: ")
            model_year = int(input("Model Yılı: "))
            milage = int(input("Kilometre: "))
            fuel_type = input("Yakıt Türü (Gasoline/Diesel/Hybrid/Electric): ")
            accident = input("Kaza Geçmişi Var mı? (Evet/Hayır): ").lower() == 'evet'
            clean_title = input("Temiz Başlık mı? (Evet/Hayır): ").lower() == 'evet'

            # Giriş verisini hazırla
            input_df = pd.DataFrame({
                'brand': [brand],
                'model': [model_name],
                'model_year': [model_year],
                'milage': [milage],
                'fuel_type': [fuel_type],
                'accident': [1 if accident else 0],
                'clean_title': [1 if clean_title else 0],
                'age': [2023 - model_year],
                'brand_model': [f"{brand}_{model_name}"]
            })

            # Tahmin yap
            price = predict_price(input_df)
            if price:
                print(f"\nTahmini Fiyat: ${price:,.2f}")
            else:
                print("Tahmin yapılamadı.")

            # Devam etmek isteyip istemediğini sor
            cont = input("\nBaşka bir tahmin yapmak ister misiniz? (Evet/Hayır): ").lower()
            if cont != 'evet':
                break

        except Exception as e:
            print(f"Hata oluştu: {str(e)}")
            print("Lütfen geçerli değerler girin.")

if __name__ == "__main__":
    main()

Toplam veri sayısı: 314223

Veri seti özeti:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314223 entries, 0 to 314222
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            314223 non-null  int64  
 1   brand         314223 non-null  object 
 2   model         314223 non-null  object 
 3   model_year    314223 non-null  int64  
 4   milage        314223 non-null  int64  
 5   fuel_type     305757 non-null  object 
 6   engine        314223 non-null  object 
 7   transmission  314223 non-null  object 
 8   ext_col       314223 non-null  object 
 9   int_col       314223 non-null  object 
 10  accident      310139 non-null  object 
 11  clean_title   278565 non-null  object 
 12  price         188533 non-null  float64
dtypes: float64(1), int64(3), object(9)
memory usage: 31.2+ MB
None

Eksik değerler:
id                   0
brand                0
model                0
model_year           0
milage 

/tmp/ipykernel_1002/3259467718.py:67: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['engine_power'].fillna(df['engine_power'].median(), inplace=True)
/tmp/ipykernel_1002/3259467718.py:68: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value

ValueError: Input y contains NaN.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Veri yükleme ve birleştirme
def load_and_combine_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Veri setlerini birleştirerek daha fazla veri ile çalışma
    combined_df = pd.concat([train_df, test_df], ignore_index=True)

    print(f"Toplam veri sayısı: {len(combined_df)}")
    print("\nVeri seti özeti:")
    print(combined_df.info())

    return combined_df

# Veri temizleme ve ön işleme
def clean_and_preprocess(df):
    # Eksik değer analizi
    print("\nEksik değerler:")
    print(df.isnull().sum())

    # Kategorik değişkenleri dönüştürme
    label_encoder = LabelEncoder()
    df['fuel_type'] = df['fuel_type'].fillna('Unknown')
    df['fuel_type'] = label_encoder.fit_transform(df['fuel_type'])

    # Kaza geçmişi - daha detaylı işleme
    df['accident'] = df['accident'].apply(
        lambda x: 1 if isinstance(x, str) and 'accident' in x.lower() else 0)

    # Temiz başlık - daha güvenli dönüşüm
    df['clean_title'] = df['clean_title'].map(
        lambda x: 1 if str(x).lower() == 'yes' else 0 if pd.notna(x) else 0)

    # Sayısal değişkenlerde aykırı değer temizleme
    numerical_cols = ['milage', 'price', 'model_year']
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Aykırı değerleri baskılama
        df[col] = np.where(df[col] < lower_bound, lower_bound,
                          np.where(df[col] > upper_bound, upper_bound, df[col]))

    # Motor bilgilerini çıkarma (daha gelişmiş)
    df['engine_power'] = df['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)
    df['engine_volume'] = df['engine'].str.extract(r'(\d+\.?\d*)L').astype(float)
    df['cylinders'] = df['engine'].str.extract(r'(\d+)\s*Cylinder').astype(float)

    # Eksik motor değerlerini ortalama ile doldurma
    df['engine_power'].fillna(df['engine_power'].median(), inplace=True)
    df['engine_volume'].fillna(df['engine_volume'].median(), inplace=True)
    df['cylinders'].fillna(df['cylinders'].median(), inplace=True)

    # Araç yaşını hesaplama
    df['age'] = 2023 - df['model_year']

    # Gereksiz sütunları çıkarma
    df.drop(['engine', 'transmission', 'ext_col', 'int_col'], axis=1, inplace=True)

    return df

# Özellik mühendisliği
def feature_engineering(df):
    # Marka-model etkileşimi
    df['brand_model'] = df['brand'] + '_' + df['model']

    # Kilometre başına fiyat (target encoding benzeri)
    brand_model_avg = df.groupby('brand_model')['price'].mean().to_dict()
    df['brand_model_avg_price'] = df['brand_model'].map(brand_model_avg)

    # Yakıt tipine göre ortalama fiyat
    fuel_avg = df.groupby('fuel_type')['price'].mean().to_dict()
    df['fuel_type_avg_price'] = df['fuel_type'].map(fuel_avg)

    # Kategorik değişkenleri one-hot encoding
    df = pd.get_dummies(df, columns=['brand', 'model'], drop_first=True)

    return df

# Model eğitimi ve değerlendirme
def train_and_evaluate(df):
    # Bağımlı ve bağımsız değişkenler
    X = df.drop(['price', 'id', 'brand_model'], axis=1)
    y = df['price']

    # Kategorik ve sayısal sütunları ayırma
    categorical_cols = [c for c in X.columns if X[c].dtype == 'object']
    numerical_cols = [c for c in X.columns if X[c].dtype in ['int64', 'float64']]

    # Ön işleme pipeline'ı
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols)])

    # Modeller
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(),
        'Lasso': Lasso(),
        'Random Forest': RandomForestRegressor(random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42)
    }

    # Model değerlendirme
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    results = []
    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        results.append({
            'Model': name,
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse,
            'R2': r2
        })

        # En iyi modeli kaydet
        if name == 'Gradient Boosting':
            joblib.dump(pipeline, 'best_model.pkl')

    # Sonuçları DataFrame'e dönüştür
    results_df = pd.DataFrame(results)
    print("\nModel Performansları:")
    print(results_df.sort_values(by='R2', ascending=False))

    # Performans görselleştirme
    plt.figure(figsize=(12, 6))
    sns.barplot(x='Model', y='R2', data=results_df)
    plt.title('Model Performans Karşılaştırması (R2 Skoru)')
    plt.xticks(rotation=45)
    plt.show()

    return results_df

# Tahmin fonksiyonu
def predict_price(input_data, model_path='best_model.pkl'):
    try:
        model = joblib.load(model_path)
        prediction = model.predict(input_data)
        return prediction[0]
    except Exception as e:
        print(f"Tahmin hatası: {str(e)}")
        return None

# Ana işlem akışı
def main():
    # Veri yükle ve işle
    df = load_and_combine_data('/content/train.csv', '/content/test.csv')
    df = clean_and_preprocess(df)
    df = feature_engineering(df)

    # Model eğit ve değerlendir
    results = train_and_evaluate(df)

    # Kullanıcı etkileşimi
    print("\nAraç Fiyat Tahmini Uygulaması")
    print("-----------------------------")

    while True:
        try:
            print("\nAraç özelliklerini girin:")
            brand = input("Marka: ")
            model_name = input("Model: ")
            model_year = int(input("Model Yılı: "))
            milage = int(input("Kilometre: "))
            fuel_type = input("Yakıt Türü (Gasoline/Diesel/Hybrid/Electric): ")
            accident = input("Kaza Geçmişi Var mı? (Evet/Hayır): ").lower() == 'evet'
            clean_title = input("Temiz Başlık mı? (Evet/Hayır): ").lower() == 'evet'

            # Giriş verisini hazırla
            input_df = pd.DataFrame({
                'brand': [brand],
                'model': [model_name],
                'model_year': [model_year],
                'milage': [milage],
                'fuel_type': [fuel_type],
                'accident': [1 if accident else 0],
                'clean_title': [1 if clean_title else 0],
                'age': [2023 - model_year],
                'brand_model': [f"{brand}_{model_name}"]
            })

            # Tahmin yap
            price = predict_price(input_df)
            if price:
                print(f"\nTahmini Fiyat: ${price:,.2f}")
            else:
                print("Tahmin yapılamadı.")

            # Devam etmek isteyip istemediğini sor
            cont = input("\nBaşka bir tahmin yapmak ister misiniz? (Evet/Hayır): ").lower()
            if cont != 'evet':
                break

        except Exception as e:
            print(f"Hata oluştu: {str(e)}")
            print("Lütfen geçerli değerler girin.")

if __name__ == "__main__":
    main()